In [1]:
import numpy as np

# 1. Definimos parámetros de negocio (ej. 600 pts a odds 20:1, y +50 pts por duplicar momios)
score_base = 600
odds_base = 20
pdo = 50

factor = pdo / np.log(2)
offset = score_base - factor * np.log(odds_base)

# 2. Calculamos Odds de ser "Bueno" (1 - PD) / PD y el Score final
# Acotamos PD ligeramente para evitar divisiones entre cero en casos extremos
pd = np.clip(df_scorecard['prob_default_modelo'], 1e-6, 1 - 1e-6)
odds_bueno = (1 - pd) / pd

df_scorecard['score'] = offset + factor * np.log(odds_bueno)

# 3. Revisamos el score promedio por rango de edad
resumen_score = df_scorecard.groupby('rango_edad', observed=False).agg(
    tasa_default_real=('default', 'mean'),
    prob_predicha=('prob_default_modelo', 'mean'),
    score_promedio=('score', 'mean')
)
print(resumen_score.round(2))

NameError: name 'df_scorecard' is not defined

In [2]:
import numpy as np
import pandas as pd

assert 'df_scorecard' in globals(), "Falta df_scorecard: ejecuta primero las celdas anteriores."
assert 'prob_default_modelo' in df_scorecard.columns, "Falta prob_default_modelo: ejecuta primero la regresión logística."

score_base = 600
odds_base = 20
pdo = 50

factor = pdo / np.log(2)
offset = score_base - factor * np.log(odds_base)

pd_default = np.clip(df_scorecard['prob_default_modelo'].astype(float), 1e-6, 1 - 1e-6)
odds_bueno = (1.0 - pd_default) / pd_default

df_scorecard['score'] = offset + factor * np.log(odds_bueno)

resumen_score = df_scorecard.groupby('rango_edad', observed=False).agg(
    tasa_default_real=('default', 'mean'),
    prob_predicha=('prob_default_modelo', 'mean'),
    score_promedio=('score', 'mean')
)
print(resumen_score.round(2))

AssertionError: Falta df_scorecard: ejecuta primero las celdas anteriores.

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Simulación de cartera con relación real entre edad y default
np.random.seed(42)
n = 5000
edad = np.random.randint(18, 70, size=n)
df_scorecard = pd.DataFrame({'edad': edad})

def prob_por_edad(e):
    if e <= 25:
        return 0.28
    elif e <= 35:
        return 0.12
    elif e <= 50:
        return 0.08
    else:
        return 0.22

df_scorecard['prob_default_real'] = df_scorecard['edad'].apply(prob_por_edad)
df_scorecard['default'] = np.random.binomial(1, df_scorecard['prob_default_real'])
df_scorecard['rango_edad'] = pd.cut(
    df_scorecard['edad'],
    bins=[17, 25, 35, 50, 70],
    labels=['18-25', '26-35', '36-50', '51-70']
)

# 2. WOE e Information Value (IV)
tabla = pd.crosstab(df_scorecard['rango_edad'], df_scorecard['default'])
tabla.columns = ['Buenos', 'Malos']
tabla['Dist_Buenos'] = tabla['Buenos'] / tabla['Buenos'].sum()
tabla['Dist_Malos'] = tabla['Malos'] / tabla['Malos'].sum()
tabla['WOE'] = np.log(tabla['Dist_Buenos'] / tabla['Dist_Malos'])
tabla['IV_parcial'] = (tabla['Dist_Buenos'] - tabla['Dist_Malos']) * tabla['WOE']
iv_total = tabla['IV_parcial'].sum()

woe_map = tabla['WOE'].to_dict()
df_scorecard['edad_woe'] = df_scorecard['rango_edad'].map(woe_map).astype(float)

# 3. Regresión Logística sobre WOE
X = df_scorecard[['edad_woe']]
y = df_scorecard['default']

modelo_log = LogisticRegression()
modelo_log.fit(X, y)

df_scorecard['prob_predicha'] = modelo_log.predict_proba(X)[:, 1]
auc = roc_auc_score(y, df_scorecard['prob_predicha'])

# 4. Escalamiento a Score con PDO (Points to Double the Odds)
score_base = 600
odds_base = 20
pdo = 50

factor = pdo / np.log(2)
offset = score_base - factor * np.log(odds_base)

pd_clip = np.clip(df_scorecard['prob_predicha'], 1e-6, 1 - 1e-6)
odds_bueno = (1.0 - pd_clip) / pd_clip
df_scorecard['score'] = offset + factor * np.log(odds_bueno)

print(f"IV de Edad: {iv_total:.4f} | ROC-AUC: {auc:.4f}\n")
resumen = df_scorecard.groupby('rango_edad', observed=False).agg(
    clientes=('default', 'count'),
    woe=('edad_woe', 'first'),
    default_real=('default', 'mean'),
    pd_predicha=('prob_predicha', 'mean'),
    score_prom=('score', 'mean')
)
print(resumen.round(4))

IV de Edad: 0.2520 | ROC-AUC: 0.6348

            clientes     woe  default_real  pd_predicha  score_prom
rango_edad                                                         
18-25            764 -0.6572        0.2723       0.2716    455.0713
26-35            919  0.3042        0.1251       0.1254    523.9906
36-50           1479  0.7596        0.0832       0.0836    556.6385
51-70           1838 -0.2487        0.1991       0.1990    484.3551


In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. Gini y KS sobre el score actual
auc = roc_auc_score(df_scorecard['default'], df_scorecard['prob_predicha'])
gini = 2 * auc - 1

df_eval = df_scorecard[['score', 'default']].copy()
df_eval['es_bueno'] = (df_eval['default'] == 0).astype(int)
df_eval['es_malo'] = (df_eval['default'] == 1).astype(int)

# Ordenamos de menor a mayor score y agrupamos en 10 deciles
df_eval['decil'] = pd.qcut(df_eval['score'], q=10, labels=False, duplicates='drop')
tabla_ks = df_eval.groupby('decil', as_index=False).agg(
    buenos=('es_bueno', 'sum'),
    malos=('es_malo', 'sum')
)
tabla_ks['cum_buenos'] = tabla_ks['buenos'].cumsum() / tabla_ks['buenos'].sum()
tabla_ks['cum_malos'] = tabla_ks['malos'].cumsum() / tabla_ks['malos'].sum()
ks_stat = np.max(np.abs(tabla_ks['cum_malos'] - tabla_ks['cum_buenos']))

print(f"Gini: {gini:.4f} | KS: {ks_stat:.4f}")

# 2. Función de PSI comparando distribución base vs nueva
def calcular_psi(base, nueva, bins=10):
    cortes = np.percentile(base, np.linspace(0, 100, bins + 1))
    cortes[0] = -np.inf
    cortes[-1] = np.inf

    p_base = np.histogram(base, bins=cortes)[0] / len(base)
    p_nueva = np.histogram(nueva, bins=cortes)[0] / len(nueva)

    # Evitamos ceros exactos para no dividir entre cero ni tomar log(0)
    p_base = np.clip(p_base, 1e-6, 1)
    p_nueva = np.clip(p_nueva, 1e-6, 1)

    psi = np.sum((p_nueva - p_base) * np.log(p_nueva / p_base))
    return psi

# Simulamos una cartera nueva con mayor proporción de jóvenes (18-25)
np.random.seed(123)
edad_nueva = np.random.choice(
    np.arange(18, 70),
    size=5000,
    p=np.array([0.04 if e <= 25 else 0.015 for e in range(18, 70)]) /
      np.sum([0.04 if e <= 25 else 0.015 for e in range(18, 70)])
)
rango_nueva = pd.cut(edad_nueva, bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
woe_nueva = rango_nueva.map(woe_map).astype(float).to_numpy().reshape(-1, 1)
pd_nueva = modelo_log.predict_proba(woe_nueva)[:, 1]
pd_nueva_clip = np.clip(pd_nueva, 1e-6, 1 - 1e-6)
score_nueva = offset + factor * np.log((1.0 - pd_nueva_clip) / pd_nueva_clip)

psi_valor = calcular_psi(df_scorecard['score'].to_numpy(), score_nueva)
print(f"PSI (Cartera Base vs Cartera Nueva): {psi_valor:.4f}")

Gini: 0.2696 | KS: 0.2227


AttributeError: 'numpy.ndarray' object has no attribute 'to_numpy'

In [5]:
rango_nueva = pd.cut(edad_nueva, bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
woe_nueva = np.asarray(rango_nueva.map(woe_map), dtype=float).reshape(-1, 1)
pd_nueva = modelo_log.predict_proba(woe_nueva)[:, 1]
pd_nueva_clip = np.clip(pd_nueva, 1e-6, 1 - 1e-6)
score_nueva = offset + factor * np.log((1.0 - pd_nueva_clip) / pd_nueva_clip)

psi_valor = calcular_psi(df_scorecard['score'].to_numpy(), score_nueva)
print(f"PSI (Cartera Base vs Cartera Nueva): {psi_valor:.4f}")

PSI (Cartera Base vs Cartera Nueva): 0.1813


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [7]:
woe_nueva_df = pd.DataFrame(woe_nueva, columns=['edad_woe'])
pd_nueva = modelo_log.predict_proba(woe_nueva_df)[:, 1]

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Simulamos cartera de 5,000 clientes con 3 variables con señal real
np.random.seed(42)
n = 5000

df_multi = pd.DataFrame({
    'edad': np.random.randint(18, 70, size=n),
    'ingreso_mensual': np.round(np.random.lognormal(mean=np.log(16000), sigma=0.55, size=n), 0),
    'uso_linea': np.clip(np.random.beta(a=2.0, b=2.5, size=n), 0.01, 0.99)  # % de crédito usado
})

# Log-odds sintético (a mayor ingreso -> menor riesgo; mayor uso de línea -> mayor riesgo; extremos de edad -> más riesgo)
efecto_edad = np.where(df_multi['edad'] <= 25, 0.65,
              np.where(df_multi['edad'] <= 35, -0.15,
              np.where(df_multi['edad'] <= 50, -0.45, 0.35)))

efecto_ingreso = -0.75 * ((np.log(df_multi['ingreso_mensual']) - np.log(16000)) / 0.55)
efecto_uso = 1.65 * (df_multi['uso_linea'] - 0.45)

log_odds = -1.55 + efecto_edad + efecto_ingreso + efecto_uso
prob_real = 1 / (1 + np.exp(-log_odds))
df_multi['default'] = np.random.binomial(1, prob_real)

# 2. Agrupamos (binning) cada variable
df_multi['bin_edad'] = pd.cut(df_multi['edad'], bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
df_multi['bin_ingreso'] = pd.qcut(df_multi['ingreso_mensual'], q=4, labels=['Q1_Bajo', 'Q2_MedioBajo', 'Q3_MedioAlto', 'Q4_Alto'])
df_multi['bin_uso'] = pd.cut(df_multi['uso_linea'], bins=[0.0, 0.25, 0.50, 0.75, 1.0], labels=['0-25%', '25-50%', '50-75%', '75-100%'])

# 3. Función WOE e IV reutilizable
def woe_iv_variable(df, col_grupo, col_target='default'):
    tab = pd.crosstab(df[col_grupo], df[col_target])
    tab.columns = ['Buenos', 'Malos']
    # Suavizado pequeño por seguridad si algún bin tuviera 0 casos
    dist_b = (tab['Buenos'] + 0.5) / (tab['Buenos'].sum() + 0.5 * len(tab))
    dist_m = (tab['Malos'] + 0.5) / (tab['Malos'].sum() + 0.5 * len(tab))
    woe = np.log(dist_b / dist_m)
    iv = np.sum((dist_b - dist_m) * woe)
    tabla_res = pd.DataFrame({
        'Buenos': tab['Buenos'],
        'Malos': tab['Malos'],
        'WOE': woe
    })
    return tabla_res, iv

vars_bins = ['bin_edad', 'bin_ingreso', 'bin_uso']
iv_resumen = {}

for col in vars_bins:
    tabla_v, iv_v = woe_iv_variable(df_multi, col)
    iv_resumen[col] = round(iv_v, 4)
    df_multi[col + '_woe'] = df_multi[col].map(tabla_v['WOE']).astype(float)
    print(f"\n--- {col} (IV = {iv_v:.4f}) ---")
    print(tabla_v.round(4))

print("\nResumen IV por variable:", iv_resumen)

# 4. Regresión Logística multivariable sobre variables WOE
cols_woe = [c + '_woe' for c in vars_bins]
X_multi = df_multi[cols_woe]
y_multi = df_multi['default']

modelo_multi = LogisticRegression()
modelo_multi.fit(X_multi, y_multi)

df_multi['pd_multi'] = modelo_multi.predict_proba(X_multi)[:, 1]
auc_multi = roc_auc_score(y_multi, df_multi['pd_multi'])
gini_multi = 2 * auc_multi - 1

print(f"\nROC-AUC Multivariable: {auc_multi:.4f} | Gini Multivariable: {gini_multi:.4f}")
for nombre, coef in zip(cols_woe, modelo_multi.coef_[0]):
    print(f"  Coeficiente {nombre}: {coef:.4f}")


--- bin_edad (IV = 0.1329) ---
          Buenos  Malos     WOE
bin_edad                       
18-25        528    236 -0.5086
26-35        747    172  0.1536
36-50       1277    202  0.5292
51-70       1389    449 -0.1842

--- bin_ingreso (IV = 0.4042) ---
              Buenos  Malos     WOE
bin_ingreso                        
Q1_Bajo          786    464 -0.7861
Q2_MedioBajo     969    282 -0.0796
Q3_MedioAlto    1045    204  0.3190
Q4_Alto         1141    109  1.0314

--- bin_uso (IV = 0.0841) ---
         Buenos  Malos     WOE
bin_uso                       
0-25%       887    152  0.4485
25-50%     1612    386  0.1157
50-75%     1117    379 -0.2327
75-100%     325    142 -0.4867

Resumen IV por variable: {'bin_edad': np.float64(0.1329), 'bin_ingreso': np.float64(0.4042), 'bin_uso': np.float64(0.0841)}

ROC-AUC Multivariable: 0.7203 | Gini Multivariable: 0.4406
  Coeficiente bin_edad_woe: -1.0943
  Coeficiente bin_ingreso_woe: -1.0393
  Coeficiente bin_uso_woe: -1.0739


In [9]:
import numpy as np
import pandas as pd

np.random.seed(42)

cohortes = ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']
mobs = [1, 2, 3, 4, 5, 6]

# Supongamos que las cohortes más recientes originaron con ligeramente mayor riesgo
riesgo_cohorte = {
    '2025-01': 0.85,
    '2025-02': 0.90,
    '2025-03': 1.00,
    '2025-04': 1.12,
    '2025-05': 1.25,
    '2025-06': 1.35
}

filas = []
for c in cohortes:
    n_creditos = 1000
    # Probabilidad acumulada de caer en 60+ días de atraso conforme crece MOB
    for mob in mobs:
        prob_mora60 = np.clip(0.012 * (mob ** 1.35) * riesgo_cohorte[c], 0.005, 0.45)
        n_mora60 = np.random.binomial(n_creditos, prob_mora60)
        filas.append({
            'cohorte': c,
            'mob': mob,
            'creditos_activos': n_creditos,
            'mora_60_mas': n_mora60,
            'tasa_mora_60_mas': n_mora60 / n_creditos
        })

df_vintage = pd.DataFrame(filas)

# 1. Matriz de Vintage (% de créditos en 60+ días de mora por cohorte y MOB)
matriz_vintage = df_vintage.pivot(index='cohorte', columns='mob', values='tasa_mora_60_mas')
print("--- MATRIZ VINTAGE: TASA DE MORA 60+ DÍAS POR COHORTE Y MOB ---")
print((matriz_vintage * 100).round(2))

# 2. Ejemplo de Roll Rate mes a mes entre cubetas de mora
estados = ['Al_dia', '1_29_dias', '30_59_dias', '60_89_dias', '90_mas_dias']
# Matriz de transición: cada fila suma 1.0 (hacia dónde se mueve el saldo/créditos al mes t+1)
matriz_transicion = np.array([
    [0.93, 0.055, 0.012, 0.002, 0.001],  # Desde Al_dia
    [0.35, 0.40,  0.18,  0.05,  0.02],   # Desde 1_29_dias
    [0.12, 0.18,  0.35,  0.25,  0.10],   # Desde 30_59_dias
    [0.05, 0.08,  0.12,  0.35,  0.40],   # Desde 60_89_dias
    [0.02, 0.03,  0.05,  0.10,  0.80]    # Desde 90_mas_dias
])

df_roll = pd.DataFrame(matriz_transicion, index=estados, columns=estados)
print("\n--- MATRIZ DE TRANSICIÓN (ROLL RATES MENSUALES) ---")
print((df_roll * 100).round(1))

roll_30_a_peor = df_roll.loc['30_59_dias', ['60_89_dias', '90_mas_dias']].sum() * 100
print(f"\nRoll rate de '30_59_dias' hacia '60+' días el siguiente mes: {roll_30_a_peor:.1f}%")

--- MATRIZ VINTAGE: TASA DE MORA 60+ DÍAS POR COHORTE Y MOB ---
mob        1    2    3     4     5     6
cohorte                                 
2025-01  0.9  3.5  5.0   6.8   7.7   9.7
2025-02  1.4  2.3  4.9   7.1  10.2  10.7
2025-03  1.0  2.3  6.0   6.4   9.6  13.8
2025-04  2.0  4.3  6.6   8.3  10.2  14.6
2025-05  1.3  3.4  5.1  10.2  13.4  15.9
2025-06  1.5  4.2  6.4   9.0  14.5  16.3

--- MATRIZ DE TRANSICIÓN (ROLL RATES MENSUALES) ---
             Al_dia  1_29_dias  30_59_dias  60_89_dias  90_mas_dias
Al_dia         93.0        5.5         1.2         0.2          0.1
1_29_dias      35.0       40.0        18.0         5.0          2.0
30_59_dias     12.0       18.0        35.0        25.0         10.0
60_89_dias      5.0        8.0        12.0        35.0         40.0
90_mas_dias     2.0        3.0         5.0        10.0         80.0

Roll rate de '30_59_dias' hacia '60+' días el siguiente mes: 35.0%


In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Reutilizamos o armamos una cartera base limpia con PD estimada y montos en MXN
np.random.seed(42)
n = 5000

df_el = pd.DataFrame({
    'edad': np.random.randint(18, 70, size=n),
    'ingreso_mensual': np.round(np.random.lognormal(mean=np.log(16000), sigma=0.55, size=n), 0),
    'linea_credito': np.round(np.random.lognormal(mean=np.log(35000), sigma=0.60, size=n), -2),
    'uso_linea': np.clip(np.random.beta(a=2.0, b=2.5, size=n), 0.05, 0.98)
})

# Asignamos una PD estimada coherente con el perfil de riesgo (o usa df_multi['pd_multi'] si ya lo corriste)
efecto_edad = np.where(df_el['edad'] <= 25, 0.65,
              np.where(df_el['edad'] <= 35, -0.15,
              np.where(df_el['edad'] <= 50, -0.45, 0.35)))
efecto_ingreso = -0.75 * ((np.log(df_el['ingreso_mensual']) - np.log(16000)) / 0.55)
efecto_uso = 1.65 * (df_el['uso_linea'] - 0.45)

log_odds = -1.55 + efecto_edad + efecto_ingreso + efecto_uso
df_el['pd'] = 1 / (1 + np.exp(-log_odds))

# 2. EAD: saldo actual + factor de disposición adicional (CCF) sobre el remanente de la línea
saldo_actual = df_el['linea_credito'] * df_el['uso_linea']
remanente_linea = df_el['linea_credito'] * (1 - df_el['uso_linea'])
ccf = 0.35  # 35% de la línea no usada que se suele disponer antes del default
df_el['ead'] = saldo_actual + ccf * remanente_linea

# 3. LGD: severidad de pérdida (ej. 55% base, ligeramente mayor si el uso de línea ya era alto)
df_el['lgd'] = np.clip(0.50 + 0.15 * df_el['uso_linea'], 0.30, 0.85)

# 4. Pérdida Esperada (EL) en pesos
df_el['el_mxn'] = df_el['pd'] * df_el['lgd'] * df_el['ead']

df_el['rango_edad'] = pd.cut(
    df_el['edad'],
    bins=[17, 25, 35, 50, 70],
    labels=['18-25', '26-35', '36-50', '51-70']
)

resumen_el = df_el.groupby('rango_edad', observed=False).agg(
    clientes=('pd', 'count'),
    pd_prom=('pd', 'mean'),
    lgd_prom=('lgd', 'mean'),
    ead_prom=('ead', 'mean'),
    el_prom_mxn=('el_mxn', 'mean'),
    el_total_mxn=('el_mxn', 'sum')
)

print(f"Pérdida Esperada Total de la cartera: ${df_el['el_mxn'].sum():,.2f} MXN")
print(f"EL promedio por cliente: ${df_el['el_mxn'].mean():,.2f} MXN\n")
print(resumen_el.round({'pd_prom': 4, 'lgd_prom': 4, 'ead_prom': 0, 'el_prom_mxn': 0, 'el_total_mxn': 0}))

Pérdida Esperada Total de la cartera: $17,723,218.37 MXN
EL promedio por cliente: $3,544.64 MXN

            clientes  pd_prom  lgd_prom  ead_prom  el_prom_mxn  el_total_mxn
rango_edad                                                                  
18-25            764   0.3094    0.5646   25846.0       4848.0     3703779.0
26-35            919   0.1816    0.5656   27337.0       2962.0     2722486.0
36-50           1479   0.1443    0.5672   26169.0       2306.0     3410372.0
51-70           1838   0.2625    0.5673   26679.0       4291.0     7886582.0


In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 5000

df_val = pd.DataFrame({
    'edad': np.random.randint(18, 70, size=n),
    'ingreso_mensual': np.round(np.random.lognormal(mean=np.log(16000), sigma=0.55, size=n), 0),
    'uso_linea': np.clip(np.random.beta(a=2.0, b=2.5, size=n), 0.01, 0.99)
})

efecto_edad = np.where(df_val['edad'] <= 25, 0.65,
              np.where(df_val['edad'] <= 35, -0.15,
              np.where(df_val['edad'] <= 50, -0.45, 0.35)))
efecto_ingreso = -0.75 * ((np.log(df_val['ingreso_mensual']) - np.log(16000)) / 0.55)
efecto_uso = 1.65 * (df_val['uso_linea'] - 0.45)

prob_real = 1 / (1 + np.exp(-(-1.55 + efecto_edad + efecto_ingreso + efecto_uso)))
df_val['default'] = np.random.binomial(1, prob_real)

# Bins fijos (reglas de corte definidas)
df_val['bin_edad'] = pd.cut(df_val['edad'], bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
df_val['bin_ingreso'] = pd.cut(
    df_val['ingreso_mensual'],
    bins=[0, 12000, 16000, 22000, np.inf],
    labels=['Hasta_12k', '12k_16k', '16k_22k', '22k_mas']
)
df_val['bin_uso'] = pd.cut(df_val['uso_linea'], bins=[0.0, 0.25, 0.50, 0.75, 1.0], labels=['0-25%', '25-50%', '50-75%', '75-100%'])

# 1. Separamos Train (70%) y Test (30%) estratificando por 'default'
df_train, df_test = train_test_split(
    df_val, test_size=0.30, random_state=42, stratify=df_val['default']
).copy(), None
df_train = df_train.copy()
df_test = train_test_split(
    df_val, test_size=0.30, random_state=42, stratify=df_val['default']
)[1].copy()

def ajustar_woe_map(train_df, col_bin, target='default'):
    tab = pd.crosstab(train_df[col_bin], train_df[target])
    tab.columns = ['Buenos', 'Malos']
    dist_b = (tab['Buenos'] + 0.5) / (tab['Buenos'].sum() + 0.5 * len(tab))
    dist_m = (tab['Malos'] + 0.5) / (tab['Malos'].sum() + 0.5 * len(tab))
    return np.log(dist_b / dist_m).to_dict()

cols_bins = ['bin_edad', 'bin_ingreso', 'bin_uso']
cols_woe = []

# 2. Calculamos WOE SOLO en Train y lo aplicamos a Train y Test
for c in cols_bins:
    mapa = ajustar_woe_map(df_train, c, 'default')
    col_w = c + '_woe'
    cols_woe.append(col_w)
    df_train[col_w] = df_train[c].map(mapa).astype(float)
    df_test[col_w] = df_test[c].map(mapa).astype(float)

# 3. Entrenamos en Train y evaluamos en Train y Test
modelo = LogisticRegression()
modelo.fit(df_train[cols_woe], df_train['default'])

pd_train = modelo.predict_proba(df_train[cols_woe])[:, 1]
pd_test = modelo.predict_proba(df_test[cols_woe])[:, 1]

auc_train = roc_auc_score(df_train['default'], pd_train)
auc_test = roc_auc_score(df_test['default'], pd_test)

gini_train = 2 * auc_train - 1
gini_test = 2 * auc_test - 1

print(f"Train -> ROC-AUC: {auc_train:.4f} | Gini: {gini_train:.4f}")
print(f"Test  -> ROC-AUC: {auc_test:.4f}  | Gini: {gini_test:.4f}")
print(f"Diferencia Gini (Train - Test): {gini_train - gini_test:+.4f}")

TypeError: list indices must be integers or slices, not str

In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Aseguramos que df_val exista y tenga sus columnas bin
assert isinstance(df_val, pd.DataFrame), "df_val debe ser un DataFrame de pandas."

# 2. Split estratificado Train (70%) / Test (30%) garantizando DataFrames
df_train, df_test = train_test_split(
    df_val,
    test_size=0.30,
    random_state=42,
    stratify=df_val['default']
)
df_train = pd.DataFrame(df_train).copy()
df_test = pd.DataFrame(df_test).copy()

def ajustar_woe_map(train_df, col_bin, target='default'):
    tab = pd.crosstab(train_df[col_bin], train_df[target])
    tab.columns = ['Buenos', 'Malos']
    dist_b = (tab['Buenos'] + 0.5) / (tab['Buenos'].sum() + 0.5 * len(tab))
    dist_m = (tab['Malos'] + 0.5) / (tab['Malos'].sum() + 0.5 * len(tab))
    return np.log(dist_b / dist_m).to_dict()

cols_bins = ['bin_edad', 'bin_ingreso', 'bin_uso']
cols_woe = []

# 3. Calculamos WOE SOLO en Train y lo aplicamos a Train y Test
for c in cols_bins:
    mapa = ajustar_woe_map(df_train, c, 'default')
    col_w = c + '_woe'
    cols_woe.append(col_w)
    df_train[col_w] = np.asarray(df_train[c].map(mapa), dtype=float)
    df_test[col_w] = np.asarray(df_test[c].map(mapa), dtype=float)

# 4. Entrenamos en Train y evaluamos en Train y Test
modelo = LogisticRegression()
modelo.fit(df_train[cols_woe], df_train['default'])

pd_train = modelo.predict_proba(df_train[cols_woe])[:, 1]
pd_test = modelo.predict_proba(df_test[cols_woe])[:, 1]

auc_train = roc_auc_score(df_train['default'], pd_train)
auc_test = roc_auc_score(df_test['default'], pd_test)

gini_train = 2 * auc_train - 1
gini_test = 2 * auc_test - 1

print(f"Train -> ROC-AUC: {auc_train:.4f} | Gini: {gini_train:.4f}")
print(f"Test  -> ROC-AUC: {auc_test:.4f}  | Gini: {gini_test:.4f}")
print(f"Diferencia Gini (Train - Test): {gini_train - gini_test:+.4f}")

Train -> ROC-AUC: 0.7097 | Gini: 0.4193
Test  -> ROC-AUC: 0.7290  | Gini: 0.4581
Diferencia Gini (Train - Test): -0.0387


In [13]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 6000

periodos = ['2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
            '2025-01', '2025-02', '2025-03']
pesos_p = np.array([0.11, 0.11, 0.12, 0.12, 0.12, 0.12, 0.10, 0.10, 0.11])
pesos_p = pesos_p / pesos_p.sum()

df_oot = pd.DataFrame({
    'periodo_orig': np.random.choice(periodos, size=n, p=pesos_p),
    'edad': np.random.randint(18, 70, size=n),
    'ingreso_mensual': np.round(np.random.lognormal(mean=np.log(16000), sigma=0.55, size=n), 0),
    'uso_linea': np.clip(np.random.beta(a=2.0, b=2.5, size=n), 0.01, 0.99)
})

# En 2025-Q1 simulamos un ligero deterioro general de cartera (+0.25 en log-odds)
shift_tiempo = np.where(df_oot['periodo_orig'].str.startswith('2025'), 0.25, 0.0)

efecto_edad = np.where(df_oot['edad'] <= 25, 0.65,
              np.where(df_oot['edad'] <= 35, -0.15,
              np.where(df_oot['edad'] <= 50, -0.45, 0.35)))
efecto_ingreso = -0.75 * ((np.log(df_oot['ingreso_mensual']) - np.log(16000)) / 0.55)
efecto_uso = 1.65 * (df_oot['uso_linea'] - 0.45)

prob_real = 1 / (1 + np.exp(-(-1.55 + efecto_edad + efecto_ingreso + efecto_uso + shift_tiempo)))
df_oot['default'] = np.random.binomial(1, prob_real)

df_oot['bin_edad'] = pd.cut(df_oot['edad'], bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
df_oot['bin_ingreso'] = pd.cut(
    df_oot['ingreso_mensual'],
    bins=[0, 12000, 16000, 22000, np.inf],
    labels=['Hasta_12k', '12k_16k', '16k_22k', '22k_mas']
)
df_oot['bin_uso'] = pd.cut(df_oot['uso_linea'], bins=[0.0, 0.25, 0.50, 0.75, 1.0], labels=['0-25%', '25-50%', '50-75%', '75-100%'])

# 1. Corte temporal: In-Time (2024) vs Out-of-Time (2025-Q1)
df_it = df_oot[df_oot['periodo_orig'].str.startswith('2024')].copy()
df_o = df_oot[df_oot['periodo_orig'].str.startswith('2025')].copy()

def map_woe_it(df_base, col_bin):
    tab = pd.crosstab(df_base[col_bin], df_base['default'])
    tab.columns = ['Buenos', 'Malos']
    dist_b = (tab['Buenos'] + 0.5) / (tab['Buenos'].sum() + 0.5 * len(tab))
    dist_m = (tab['Malos'] + 0.5) / (tab['Malos'].sum() + 0.5 * len(tab))
    return np.log(dist_b / dist_m).to_dict()

cols_bins = ['bin_edad', 'bin_ingreso', 'bin_uso']
cols_woe = []

for c in cols_bins:
    mapa = map_woe_it(df_it, c)
    col_w = c + '_woe'
    cols_woe.append(col_w)
    df_it[col_w] = np.asarray(df_it[c].map(mapa), dtype=float)
    df_o[col_w] = np.asarray(df_o[c].map(mapa), dtype=float)

modelo_oot = LogisticRegression()
modelo_oot.fit(df_it[cols_woe], df_it['default'])

df_it['pd_pred'] = modelo_oot.predict_proba(df_it[cols_woe])[:, 1]
df_o['pd_pred'] = modelo_oot.predict_proba(df_o[cols_woe])[:, 1]

gini_it = 2 * roc_auc_score(df_it['default'], df_it['pd_pred']) - 1
gini_oot = 2 * roc_auc_score(df_o['default'], df_o['pd_pred']) - 1

# PSI sobre PD predicha (In-Time vs Out-of-Time)
cortes = np.percentile(df_it['pd_pred'], np.linspace(0, 100, 11))
cortes[0], cortes[-1] = -np.inf, np.inf
p_it = np.clip(np.histogram(df_it['pd_pred'], bins=cortes)[0] / len(df_it), 1e-6, 1)
p_oot = np.clip(np.histogram(df_o['pd_pred'], bins=cortes)[0] / len(df_o), 1e-6, 1)
psi_oot = np.sum((p_oot - p_it) * np.log(p_oot / p_it))

print(f"In-Time  (2024)    -> Gini: {gini_it:.4f} | Default real: {df_it['default'].mean():.2%} | PD predicha: {df_it['pd_pred'].mean():.2%}")
print(f"Out-of-Time (2025Q1) -> Gini: {gini_oot:.4f} | Default real: {df_o['default'].mean():.2%} | PD predicha: {df_o['pd_pred'].mean():.2%}")
print(f"PSI (IT vs OOT): {psi_oot:.4f}\n")

# 2. Tabla de calibración por quintiles de PD en Out-of-Time
df_o['quintil_pd'] = pd.qcut(df_o['pd_pred'], q=5, labels=['Q1_MenorRiesgo', 'Q2', 'Q3', 'Q4', 'Q5_MayorRiesgo'])
calibracion_oot = df_o.groupby('quintil_pd', observed=False).agg(
    casos=('default', 'count'),
    pd_predicha=('pd_pred', 'mean'),
    default_real=('default', 'mean')
)
calibracion_oot['brecha_pp'] = (calibracion_oot['default_real'] - calibracion_oot['pd_predicha']) * 100
print("--- CALIBRACIÓN EN OUT-OF-TIME (2025-Q1) ---")
print(calibracion_oot.round({'pd_predicha': 4, 'default_real': 4, 'brecha_pp': 2}))

In-Time  (2024)    -> Gini: 0.4411 | Default real: 21.09% | PD predicha: 21.09%
Out-of-Time (2025Q1) -> Gini: 0.4337 | Default real: 24.11% | PD predicha: 21.21%
PSI (IT vs OOT): 0.0101

--- CALIBRACIÓN EN OUT-OF-TIME (2025-Q1) ---
                casos  pd_predicha  default_real  brecha_pp
quintil_pd                                                 
Q1_MenorRiesgo    366       0.0626        0.0820       1.94
Q2                361       0.1240        0.1413       1.73
Q3                368       0.1829        0.2092       2.64
Q4                369       0.2753        0.3008       2.55
Q5_MayorRiesgo    340       0.4297        0.4882       5.86
